<a href="https://colab.research.google.com/github/bpayton0101/540-ML-Design-Solutions/blob/main/S2S_CDR_SOW_Test_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



Loading just the 30-page SOW is a much better test than loading the entire contract right now.

## LAPD SOW Test
This is a perfect test for a few reasons:

*  Focused Scope: The SOW is the "guts" of the contract. All of your prompts (SOW analysis, internal summary, and CDR) heavily rely on it. This test will show you how well the model extracts the most important information.

*  Test for "Hallucination": This is the most important part. Your SOW probably doesn't contain "Contract Signature Date" or "Payment Milestones." This is a good thing for this test. You get to see if the model correctly writes "Not Specified" for those fields. If it does, your model is accurate. If it makes something up, we need to tweak the prompt.

* Manageable Size: 30 pages is a real-world test (unlike a 1-page snippet) but still small enough to run quickly.


Cell 1: Setup and API Key 🔑

In [ ]:
# Install necessary Python libraries
!pip install -q -U google-generativeai PyPDF2

# Import libraries
import google.generativeai as genai
import PyPDF2
from google.colab import userdata

# --- Configure the Gemini API ---
# (Remember to add your API key to Colab Secrets as 'GEMINI_API_KEY')
try:
    api_key = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=api_key)
    print("✅ Gemini API configured successfully.")
except userdata.SecretNotFoundError as e:
    print("🛑 Secret not found. Please add 'GEMINI_API_KEY' to your Colab Secrets.")
except Exception as e:
    print(f"An error occurred: {e}")

# List available models to check which ones support generateContent
print("\nAvailable Generative Models:")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(f"  - {m.name}")

# Initialize the Gemini Pro model
# Changed to gemini-pro-latest as previous models encountered NotFound errors.
model = genai.GenerativeModel('gemini-pro-latest')
print("✅ Model initialized to gemini-pro-latest.")

An error occurred: Requesting secret GEMINI_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

Available Generative Models:


ReadTimeout: HTTPConnectionPool(host='localhost', port=43655): Read timed out. (read timeout=60.0)

Cell 2: PDF Extraction Helper Function 📄

In [ ]:
def extract_text_from_pdf(pdf_file_path):
    """Opens a PDF file and returns all its text content."""
    text = ""
    try:
        with open(pdf_file_path, 'rb') as pdf_file:
            pdf_reader = PyPDF2.PdfReader(pdf_file)
            for page in pdf_reader.pages:
                text += page.extract_text()
        return text
    except FileNotFoundError:
        return f"Error: The file '{pdf_file_path}' was not found. Please upload it."
    except Exception as e:
        return f"An error occurred: {e}"

Cell 3 (Modified for LAPD SOW Upload)

In [ ]:
# --- Step 3 (Modified for Single PDF SOW Upload) ---

# 1. Upload your 30-page SOW PDF to Colab
#    (e.g., 'LAPD_SOW.pdf')

# 2. Define the filename of the PDF you uploaded
snippet_pdf_file = 'LAPD_SOW.pdf'  # <-- CHANGE THIS FILENAME

# 3. Extract text from your uploaded PDF
print(f"⏳ Extracting text from {snippet_pdf_file}...")
snippet_text = extract_text_from_pdf(snippet_pdf_file)

# 4. Assign this text to the main 'comprehensive_text' variable
comprehensive_text = f"""
--- DOCUMENT START: BASELINE CONTRACT ---
{snippet_text}
--- DOCUMENT END: BASELINE CONTRACT ---
"""

print(f"✅ Text from {snippet_pdf_file} is ready for analysis.")

⏳ Extracting text from LAPD_SOW.pdf...
✅ Text from LAPD_SOW.pdf is ready for analysis.


Cell 4: Define All Analysis Functions 🧠

In [ ]:
def extract_sow_and_risk(contract_text):
    """Extracts the SOW and assesses key risks."""
    prompt = f"""
    Analyze the following contract document(s). Perform two tasks:
    1.  **Extract Statement of Work (SOW):** Summarize the primary deliverables, responsibilities, and scope of the project.
    2.  **Assess Key Risks:** Identify the top 3-5 risks related to project delivery, scope creep, payment terms, or customer language. For each risk, quote the problematic clause and explain the potential impact.

    Here is the contract:
    ---
    {contract_text}
    ---
    """
    response = model.generate_content(prompt)
    return response.text

def generate_internal_summary(contract_text):
    """Generates a structured internal project summary."""
    prompt = f"""
    You are a senior Technical Program Manager. Your task is to create a concise, scannable 'Internal Project Summary' from the provided contract documents. This summary is for internal review by Project Managers and Leadership.

    Based on the examples from the Denver and Carlsbad projects, extract the following information. If a piece of information is not found, state 'Not Specified'.

    **Project Code:** (e.g., USCO24D022SS)
    **Key Dates:**
      - Contract Signature:
      - Project Kick Off:
      - Go-Live Target:
    **Core Products & Services:** (List the main components like P1 CAD, CC Aware, etc.)
    **Key Integrations:** (List 3rd-party systems like Astro, Rave, Vesta, etc.)
    **Hardware Requirements:** (List servers, etc.)
    **Interfaces:** (List data feeds like NetRMS CFS, etc.)
    **Key Risks & Special Notes:** (Summarize any payment deferrals, customer language risks, or major scope issues.)
    **Change Orders:** (Note any items explicitly identified as coming from a change order.)

    Here are the contract documents:
    ---
    {contract_text}
    ---
    """
    response = model.generate_content(prompt)
    return response.text

def generate_external_cdr(contract_text):
    """
    Generates a customer-facing 'Contract Design Review' (CDR) document
    based on the provided internal CDR process and objectives.
    """
    prompt = f"""
    You are a Senior Project Manager. Your task is to generate a formal 'Contract Design Review' (CDR) document.
    This document is for external sharing with the customer to review contracted products, SOWs, and obligations to reach a common, comprehensive understanding of the implementation scope.

    Using the contract documents provided, extract the following information and structure it as a formal CDR agenda.

    **Objective:** To review contracted products with the Customer that make up the solution as sold, including the bill of materials, all applicable project plans, and the contractual obligations of all parties in order to reach a common and comprehensive understanding of the implementation scope and effort.

    **CDR AGENDA & REVIEW POINTS:**

    **1. Purchased Products & Capabilities:**
       - [List all purchased products, software, and hardware from the Bill of Materials. Example: 'P1 CAD', 'CC Aware', 'HP Data Center Servers'.]

    **2. Statement of Work (SOW) Review:**
       - [Summarize the high-level scope, deliverables, and responsibilities as defined in the contract.]

    **3. Interfaces & Integrations Review:**
       - [List all known interfaces, integrations, and third-party solutions required for the project. Example: 'Astro', 'Rave', 'NetRMS CFS data feed'.]

    **4. Bill of Materials (BOM) & Installation Requirements:**
       - [List specific hardware and any known installation requirements mentioned.]

    **5. Change Order Review:**
       - [List any capabilities or products that were explicitly added or modified via a Change Order.]

    **6. Key Assumptions & Discovered Items for Discussion:**
       - [Identify any ambiguous language, 'TBD' items, or potential dependencies from the contract that require customer clarification.]

    **7. Next Steps:**
       - Conclude with a placeholder for the Project Manager to complete a set of CDR meeting minutes, including any actions, owners, and timelines.

    ---
    **CONTRACT DOCUMENTS FOR ANALYSIS:**
    {contract_text}
    ---
    """
    response = model.generate_content(prompt)
    return response.text

Cell 5: Run SOW & Risk Analysis 🚀

In [ ]:
print("⏳ Analyzing SOW and Risks...")
sow_risk_report = extract_sow_and_risk(comprehensive_text)
print("--- SOW & RISK REPORT ---")
print(sow_risk_report)

⏳ Analyzing SOW and Risks...


ERROR:tornado.access:503 POST /v1beta/models/gemini-pro-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 6382.97ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-pro-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 456.39ms


ReadTimeout: HTTPConnectionPool(host='localhost', port=43655): Read timed out. (read timeout=591.4020888805389)

Cell 6: Run Internal Summary Generation 🚀

In [ ]:
print("\n⏳ Generating Internal Project Summary...")
internal_summary = generate_internal_summary(comprehensive_text)
print("\n--- INTERNAL PROJECT SUMMARY ---")
print(internal_summary)


⏳ Generating Internal Project Summary...


ERROR:tornado.access:503 POST /v1beta/models/gemini-pro-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 938.98ms



--- INTERNAL PROJECT SUMMARY ---
Here is the Internal Project Summary based on the provided documents.

***

### **Internal Project Summary: Carlsbad (USCA23D266SS)**

**Project Code:** USCA23D266SS

**Key Dates:**
*   **Contract Signature:** 2/6/2025
*   **Project Kick Off:** Not Specified
*   **Go-Live Target:** Not Specified (Note: Contract includes LDs if project finishes *before* 20 months.)

**Core Products & Services:**
*   P1 CAD with ARL (On-Premise) & CAD Continuity DR
*   CC Aware for RTCC
*   C3
*   ActiveEye Managed Detection
*   P1 Mobile with Mapping (Windows, iOS, Android)

**Key Integrations:**
*   Vesta 911
*   Vigilant ALPR
*   Astro PTT
*   Rapid SOS

**Hardware Requirements:**
*   MSI-provided Managed Hardware (MSI responsible for backups, upgrades, OS patching, etc. per Exhibit A).

**Interfaces:**
*   NetRMS CFS data feed & Query
*   SPIDR Tech CFS data feed
*   Axon CFS data feed
*   CLETS/NCIC State Query (via CommSys)
*   Crossroads Accident & Citations Query

Cell 7: Run External CDR Generation 🚀

In [ ]:
print("\n⏳ Generating Customer-Facing Contract Design Review (CDR)...")
external_cdr = generate_external_cdr(comprehensive_text)
print("\n--- EXTERNAL CONTRACT DESIGN REVIEW ---")
print(external_cdr)


⏳ Generating Customer-Facing Contract Design Review (CDR)...

--- EXTERNAL CONTRACT DESIGN REVIEW ---
Of course. As a Senior Project Manager, here is the formal Contract Design Review (CDR) document based on the provided contract summary.

---

### **Contract Design Review (CDR)**

**Project Name:** Carlsbad Public Safety Solution Implementation (USCA23D266SS)
**Date:** [Date of Meeting]
**Attendees:**
*   **Customer (Carlsbad):** [List Attendee Names/Roles]
*   **Motorola Solutions:** [List Attendee Names/Roles]

**Objective:** To review contracted products with the Customer that make up the solution as sold, including the bill of materials, all applicable project plans, and the contractual obligations of all parties in order to reach a common and comprehensive understanding of the implementation scope and effort.

---

### **CDR AGENDA & REVIEW POINTS**

**1. Purchased Products & Capabilities:**

The following products and capabilities have been purchased as part of the on-premise, 